# 300장 실험 준비 (채점표 + 검색 DB 만들기)

위에서부터 **순서대로** 실행하세요. 바꿔야 하는 곳은 **2번 셀의 zip 경로**뿐이에요.

| 순서 | 하는 일 |
|---|---|
| 1 | 구글 드라이브 연결 + 코드 최신으로 받기 |
| 2 | zip 경로 적기 (★ 여기만 수정) |
| 3 | zip 풀기 |
| 4 | 채점표 `manifest.csv` 만들기 |
| 5 | 검색 DB `search_db.csv` 만들기 |
| 6 | 결과 미리보기 |
| 7 | 결과를 드라이브에 저장 (런타임이 끊겨도 남게) |

In [ ]:
# 1) 구글 드라이브 연결 + 코드 최신으로 받기
from google.colab import drive
drive.mount('/content/drive')

import os
if os.path.exists('/content/dl_deep_imgps'):
    %cd /content/dl_deep_imgps
    !git pull
else:
    %cd /content
    !git clone https://github.com/yewon51019/dl_deep_imgps
    %cd /content/dl_deep_imgps
!ls exp300

In [ ]:
# 2) ★ zip 파일 경로 적기 (드라이브 왼쪽 파일창에서 zip 우클릭 -> '경로 복사' 후 붙여넣기)
ZIP_FILES = [
    '/content/drive/MyDrive/약제각인인식_테스트셋.zip',
    '/content/drive/MyDrive/약제각인인식_테스트셋(1).zip',
]

# (선택) AI Hub 전체 라벨 폴더가 드라이브에 있으면 적기. 없으면 빈 칸 [] 그대로 두기.
#   -> 검색 DB 후보를 27종이 아니라 실제처럼 많게 만들 때 사용
EXTRA_LABEL_DIRS = []

# 개발용 비율 (0.2 = 20%, 0 이면 나누지 않음)
DEV_RATIO = 0.2

for z in ZIP_FILES:
    print('있음 ✓' if os.path.exists(z) else '없음 ✗ (경로 확인!)', z)

In [ ]:
# 3) zip 풀기 (zip 마다 다른 폴더에)
import subprocess
UNZIP_DIRS = []
for i, z in enumerate(ZIP_FILES, 1):
    out = f'/content/data{i}'
    if not os.path.exists(out):
        subprocess.run(['unzip', '-q', '-o', z, '-d', out], check=True)
    UNZIP_DIRS.append(out)

# images/ 와 labels/ 가 같이 들어있는 폴더를 자동으로 찾기
ROOTS = []
for d in UNZIP_DIRS:
    for cur, subdirs, files in os.walk(d):
        if 'images' in subdirs and 'labels' in subdirs:
            ROOTS.append(cur)
print('찾은 폴더:'); [print(' ', r) for r in ROOTS]

In [ ]:
# 4) 채점표 manifest.csv 만들기 (이미지 1장당 1줄, 정답 포함)
subprocess.run(['python', 'exp300/make_manifest.py', '--roots', *ROOTS,
                '--out', '/content/manifest.csv', '--dev-ratio', str(DEV_RATIO)])

In [ ]:
# 5) 검색 DB search_db.csv 만들기 (약 1종당 1줄)
label_dirs = [os.path.join(r, 'labels') for r in ROOTS] + EXTRA_LABEL_DIRS
subprocess.run(['python', 'exp300/make_search_db.py', '--label-roots', *label_dirs,
                '--out', '/content/search_db.csv'])

In [ ]:
# 6) 결과 미리보기
import pandas as pd
m = pd.read_csv('/content/manifest.csv')
db = pd.read_csv('/content/search_db.csv')
print('채점표', m.shape, '/ 검색 DB', db.shape)
display(m[['file_name','drug_N','dl_name','form','drug_dir','camera_la','rot_deg','truth','split']].head(10))
display(db[['drug_N','dl_name','form','shape','color1','print_front','print_back']])

In [ ]:
# 7) 결과를 드라이브에 저장 (런타임이 끊겨도 남도록)
SAVE_DIR = '/content/drive/MyDrive/exp300_results'
os.makedirs(SAVE_DIR, exist_ok=True)
!cp /content/manifest.csv /content/search_db.csv "{SAVE_DIR}"
!ls -l "{SAVE_DIR}"